# 02 · Bronze

Landing CSV → **Delta table** at `data/bronze/enem_raw`, partitioned by `NU_ANO`.

All columns are kept **as strings** on purpose: Bronze is a faithful copy of the source,
so a malformed value never breaks ingestion. Typing happens in Silver.
Two audit columns are added: `_ingestion_timestamp` and `_source_file`.

In [ ]:
%run ./00_setup.ipynb

In [ ]:
from pyspark.sql.functions import col, current_timestamp

df_raw = (
    spark.read.option("header", "true")
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .csv(str(LANDING_CSV))
)

df_bronze = (
    df_raw.withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))  # Spark's hidden file metadata column
)

In [ ]:
df_bronze.write.format("delta").mode("overwrite").partitionBy("NU_ANO").save(BRONZE_TABLE)

Read the table back from storage.

In [ ]:
bronze = spark.read.format("delta").load(BRONZE_TABLE)
print(f"Bronze: {bronze.count()} rows | {len(bronze.columns)} columns")
bronze.select("NU_INSCRICAO", "NU_ANO", "SG_UF_PROVA", "NU_NOTA_MT", "_ingestion_timestamp", "_source_file").show(5, truncate=False)

Every write is a commit in the Delta transaction log (`_delta_log/`).

In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{BRONZE_TABLE}`").select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

In [ ]:
spark.stop()